# EDA — LexGLUE SCOTUS

Label distribution, document length distribution, and imbalance stats that drive the
truncation and class-weighting decisions used later in training.

Run this from the repo root so `src` is importable (e.g. `jupyter notebook` from
`domain_adapted_slm/`, or add the repo root to `sys.path`).

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np

from src.data.load import label_names, load_scotus

dataset = load_scotus()
labels = label_names()
dataset

In [ ]:
# Label distribution (train split) — expect real imbalance across the 14 issue areas.
train_labels = np.array(dataset["train"]["label"])
counts = np.bincount(train_labels, minlength=len(labels))

for name, count in sorted(zip(labels, counts), key=lambda x: -x[1]):
    print(f"{name:>22}: {count:5d}")

print(f"\nimbalance ratio (max/min): {counts.max() / max(counts.min(), 1):.1f}x")

In [ ]:
plt.figure(figsize=(10, 5))
order = np.argsort(-counts)
plt.bar(range(len(labels)), counts[order])
plt.xticks(range(len(labels)), [labels[i] for i in order], rotation=60, ha="right")
plt.ylabel("train examples")
plt.title("SCOTUS label distribution (train)")
plt.tight_layout()
plt.show()

In [ ]:
# Document length (words, as a cheap proxy before tokenizer choice is finalized).
lengths = [len(t.split()) for t in dataset["train"]["text"]]
lengths = np.array(lengths)

print(f"median words: {np.median(lengths):.0f}")
print(f"p90 words:    {np.percentile(lengths, 90):.0f}")
print(f"p99 words:    {np.percentile(lengths, 99):.0f}")
print(f"max words:    {lengths.max()}")

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=60)
plt.axvline(np.median(lengths), color="black", linestyle="--", label="median")
plt.xlabel("words per document")
plt.legend()
plt.title("SCOTUS document length distribution")
plt.show()

## Truncation decision

SCOTUS opinions are long relative to Phi-3-mini's 4k-token context window. We truncate to
the first `max_length` tokens (see `configs/scotus_phi3.yaml`) rather than chunking or
summarizing — a documented limitation, not solved in week 1. Confirm the chosen `max_length`
against the percentiles above before locking in the training config.